# Gemma 4 native video captioning — T4 benchmark

Answers the question the research raised: **does Gemma 4's native video input beat the
frame-sampling path aerosearch uses today, and does it run on a Kaggle T4?**

Self-contained: no repo clone, no tunnel, no gated download (Gemma 4 is Apache 2.0, so no
`HF_TOKEN`). Clips are fetched from public URLs; one silent clip is synthesised with ffmpeg to
stand in for the "silent Macarena" case that motivated the rebuild.

**T4 note:** Turing (sm_75) has no bf16. All compute dtypes here are fp16.

Kaggle returns logs only *after* the run finishes, so every stage prints its own markers and
failures are caught rather than aborting the notebook.


In [ ]:
# --- Cell 1: preflight ---
import subprocess, sys, platform
print("python", platform.python_version())
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
print("bf16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else "n/a")


In [ ]:
# --- Cell 2: deps ---
# Gemma 4 needs a recent transformers; torchvision+librosa back the video/audio paths.
%pip install -q -U transformers accelerate bitsandbytes torchvision librosa
import transformers; print("transformers", transformers.__version__)
try:
    from transformers import AutoModelForMultimodalLM
    MM_CLS, MM_NAME = AutoModelForMultimodalLM, "AutoModelForMultimodalLM"
except ImportError:
    from transformers import AutoModelForImageTextToText as _F
    MM_CLS, MM_NAME = _F, "AutoModelForImageTextToText (fallback)"
print("model class:", MM_NAME)


In [ ]:
# --- Cell 3: clips ---
# Two real clips with motion + one synthetic silent clip.
import os, subprocess, urllib.request
os.makedirs("/kaggle/working/clips", exist_ok=True)
CLIPS = {}
for name, url in {
    "blazes": "https://github.com/bebechien/gemma/raw/refs/heads/main/videos/ForBiggerBlazes.mp4",
    "escapes": "https://github.com/bebechien/gemma/raw/refs/heads/main/videos/ForBiggerEscapes.mp4",
}.items():
    p = f"/kaggle/working/clips/{name}.mp4"
    try:
        urllib.request.urlretrieve(url, p); CLIPS[name] = p
        print("ok", name, os.path.getsize(p)//1024, "KB")
    except Exception as e:
        print("FAIL", name, repr(e))

# Silent, motion-only clip: the "silent dance" case audio-only indexing missed.
p = "/kaggle/working/clips/silent_motion.mp4"
r = subprocess.run(["ffmpeg","-y","-f","lavfi","-i",
                    "testsrc=duration=8:size=640x480:rate=10","-pix_fmt","yuv420p",p],
                   capture_output=True, text=True)
if os.path.exists(p): CLIPS["silent_motion"] = p; print("ok silent_motion (no audio track)")
else: print("FAIL ffmpeg", r.stderr[-300:])
print("clips:", list(CLIPS))


In [ ]:
# --- Cell 4: load Gemma 4 E4B at 4-bit (fp16 compute for T4) ---
import time, torch
from transformers import AutoProcessor, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E4B-it"
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_compute_dtype=torch.float16,   # T4: no bf16
                           bnb_4bit_use_double_quant=True)
t0 = time.time(); model = processor = None; LOAD_ERR = None
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = MM_CLS.from_pretrained(MODEL_ID, quantization_config=quant,
                                   device_map="auto").eval()
    print(f"loaded in {time.time()-t0:.1f}s")
    print("footprint GB:", round(model.get_memory_footprint()/1e9, 2))
except Exception as e:
    LOAD_ERR = repr(e); print("LOAD FAILED:", LOAD_ERR[:1500])


In [ ]:
# --- Cell 5: helper ---
import time, torch

VIDEO_PROMPT = ("Describe this clip for a search index: what is visibly happening, "
                "the setting, and any on-screen text. Two sentences, no preamble.")

def gen(content, **proc_kw):
    """Run one multimodal turn; returns (text, seconds, n_input_tokens) or (error, -1, -1)."""
    msgs = [{"role": "user", "content": content}]
    try:
        t = time.time()
        inputs = processor.apply_chat_template(
            msgs, tokenize=True, return_dict=True, return_tensors="pt",
            add_generation_prompt=True, **proc_kw).to(model.device)
        n_in = inputs["input_ids"].shape[-1]
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        txt = processor.decode(out[0][n_in:], skip_special_tokens=True).strip()
        return txt, time.time()-t, n_in
    except Exception as e:
        return f"ERROR: {e!r}", -1.0, -1


In [ ]:
# --- Cell 6: A/B — native video vs frame sampling (today's aerosearch path) ---
import json, subprocess, glob, os
RESULTS = {}

if model is None:
    print("skipped: model did not load")
else:
    for name, path in CLIPS.items():
        print("="*70); print("CLIP:", name); print("="*70)
        RESULTS[name] = {}

        # (a) NATIVE VIDEO: one video object, model samples frames itself at 1 fps.
        txt, secs, n_in = gen(
            [{"type": "video", "video": path}, {"type": "text", "text": VIDEO_PROMPT}],
            do_sample_frames=True, fps=1, num_frames=16, max_soft_tokens=140)
        RESULTS[name]["native_video"] = {"caption": txt, "sec": round(secs,1), "in_tokens": n_in}
        print(f"\n[native video] {secs:.1f}s, {n_in} input tokens\n{txt}\n")

        # (b) FRAME SAMPLING: 6 stills as independent images == aerosearch today
        #     (AEROSEARCH_NUM_FRAMES=6, no temporal information between them).
        fdir = f"/kaggle/working/frames_{name}"; os.makedirs(fdir, exist_ok=True)
        subprocess.run(["ffmpeg","-y","-i",path,"-vf","fps=6/8,scale=-2:480",
                        "-frames:v","6",f"{fdir}/f_%02d.png"], capture_output=True)
        frames = sorted(glob.glob(f"{fdir}/f_*.png"))[:6]
        if frames:
            from PIL import Image
            content = [{"type":"image","image":Image.open(f).convert("RGB")} for f in frames]
            content.append({"type":"text","text":VIDEO_PROMPT})
            txt2, secs2, n_in2 = gen(content, max_soft_tokens=140)
            RESULTS[name]["frames_6"] = {"caption": txt2, "sec": round(secs2,1), "in_tokens": n_in2}
            print(f"[6 sampled frames] {secs2:.1f}s, {n_in2} input tokens\n{txt2}\n")
        else:
            print("[6 sampled frames] ffmpeg produced no frames")


In [ ]:
# --- Cell 7: summary ---
import json
print("MODEL:", MODEL_ID, "| load error:", LOAD_ERR)
print(json.dumps(RESULTS, indent=2, ensure_ascii=False))
with open("/kaggle/working/gemma4_video_benchmark.json","w") as fh:
    json.dump({"model": MODEL_ID, "load_error": LOAD_ERR, "results": RESULTS}, fh,
              indent=2, ensure_ascii=False)

print("\n--- TIMING TABLE ---")
print(f"{'clip':<16}{'mode':<16}{'sec':>8}{'in_tok':>9}")
for c, modes in RESULTS.items():
    for m, d in modes.items():
        print(f"{c:<16}{m:<16}{d['sec']:>8}{d['in_tokens']:>9}")
print("\nDONE")
